# Week 61, Day 2: Independent component analysis: unmixing recordings

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w061/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: FastICA by deflation

Whiten the three mixed channels, then complete fastica(Z, k): for each component, start from a random unit vector and iterate the fixed-point update with deflation and normalisation until it converges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
gi = np.random.default_rng(612)
T = 3000
t = np.arange(T) / 1000
spikes = (gi.random(T) < 0.01) * gi.uniform(2, 4, T)
spikes = np.convolve(spikes, np.exp(-np.arange(30) / 6), mode="same")      # a sparse train of events
theta = np.sin(2 * np.pi * 7 * t)                                         # a 7 Hz oscillation
drift = ((t * 0.9) % 1) * 2 - 1                                           # a slow sawtooth artefact
S_true = np.vstack([spikes, theta, drift])
S_true = (S_true - S_true.mean(1, keepdims=True)) / S_true.std(1, keepdims=True)
A_mix = gi.normal(size=(3, 3))
Xm = A_mix @ S_true

def whiten(X):
    Xc = X - X.mean(1, keepdims=True)
    ev, E = np.linalg.eigh(np.cov(Xc))
    V = E @ np.diag(ev ** -0.5) @ E.T
    return V @ Xc, V

def fastica(Z, k, g, iters=200):
    W = np.zeros((k, Z.shape[0]))
    for c in range(k):
        w = g.normal(size=Z.shape[0]); w /= np.linalg.norm(w)
        for _ in range(iters):
            u = w @ Z
            w_new = ...
            w_new -= W[:c].T @ (W[:c] @ w_new)          # deflation
            w_new /= np.linalg.norm(w_new)
            converged = abs(abs(w_new @ w) - 1) < 1e-10
            w = w_new
            if converged:
                break
        W[c] = w
    return W

Zw, Vw = whiten(Xm)
W_ica = fastica(Zw, 3, gi)
S_hat = W_ica @ Zw
C = np.abs(np.corrcoef(np.vstack([S_true, S_hat]))[:3, 3:])
print("absolute correlation, true sources (rows) against components (columns):"); print(np.round(C, 3))
fig, ax = plt.subplots(3, 2, figsize=(9, 4), sharex=True)
for i in range(3):
    ax[i, 0].plot(t[:1000], Xm[i, :1000], lw=0.7); ax[i, 1].plot(t[:1000], S_hat[i, :1000], lw=0.7, color="C1")
ax[0, 0].set_title("recorded channels"); ax[0, 1].set_title("independent components"); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_white():
    "the whitened data have identity covariance"
    assert np.allclose(np.cov(Zw), np.eye(3), atol=1e-6)
def test_unmix():
    "each true source is recovered with absolute correlation above 0.95"
    C = np.abs(np.corrcoef(np.vstack([S_true, S_hat]))[:3, 3:])
    assert np.all(C.max(1) > 0.95)

test_white()
test_unmix()
print("All checks passed")

### Exercise: Gaussian sources cannot be unmixed

Mix two Gaussian sources with a fixed random matrix and run FastICA from 10 random starts, recording each time the best absolute correlation of the first source with a component. Do the same for two non-Gaussian sources (the event train and the oscillation).

In [ ]:
gg = np.random.default_rng(6121)
A2 = gg.normal(size=(2, 2))
S_g = gg.normal(size=(2, T))
S_ng = S_true[:2]
match_gauss, match_ng = [], []
for start in range(10):
    for S_src, store in [(S_g, match_gauss), (S_ng, match_ng)]:
        Z2, _ = whiten(A2 @ S_src)
        Y = fastica(Z2, 2, np.random.default_rng(start)) @ Z2
        store.append(...)
print("Gaussian sources, best |r| per start:    ", np.round(match_gauss, 2))
print("non-Gaussian sources, best |r| per start:", np.round(match_ng, 2))

In [ ]:
import numpy as np
def test_gauss():
    "with two Gaussian sources, the recovered match varies widely across starts (some below 0.9)"
    assert min(match_gauss) < 0.9
def test_nongauss():
    "with the non-Gaussian sources, every start matches above 0.95"
    assert min(match_ng) > 0.95

test_gauss()
test_nongauss()
print("All checks passed")

### Exercise: Cleaning blinks from six channels

Six EEG channels mix an alpha rhythm, a slower rhythm, three noise sources and a blink artefact. Unmix with FastICA, find the component with the largest kurtosis, zero it, and reconstruct the channels. Measure how much blink remains (regression of the cleaned channels on the true blink, relative to before) and how well the alpha rhythm's contribution is preserved.

In [ ]:
ge = np.random.default_rng(6122)
t6 = np.arange(5000) / 250
alpha = np.sin(2 * np.pi * 10 * t6 + 0.3 * np.cumsum(ge.normal(size=5000)) / 30)
slow = np.sin(2 * np.pi * 3 * t6) * (1 + 0.5 * np.sin(2 * np.pi * 0.2 * t6))
blink = np.zeros(5000)
for onset in ge.choice(4800, 12, replace=False):
    blink[onset:onset + 100] += 8 * np.hanning(100)
noise = ge.laplace(size=(3, 5000))
S6 = np.vstack([alpha, slow, noise, blink])
A6 = ge.normal(size=(6, 6)); A6[:, 5] = np.abs(A6[:, 5]) * [3, 2.5, 1, 0.5, 0.3, 0.2]   # blinks are strongest at frontal channels
X6 = A6 @ S6

Z6, V6 = whiten(X6)
W6 = fastica(Z6, 6, ge)
Y6 = W6 @ Z6
kurt = (Y6 ** 4).mean(1) - 3
blink_comp = int(np.argmax(kurt))
Y_clean = Y6.copy(); Y_clean[blink_comp] = 0
A_hat = np.linalg.inv(W6 @ V6)
X_clean = ... + X6.mean(1, keepdims=True)

def share(Xs, src):
    """fraction of each channel's variance explained by regression on one source, averaged over channels"""
    return np.mean([np.corrcoef(x, src)[0, 1] ** 2 for x in Xs])
residual_blink = share(X_clean, blink) / share(X6, blink)
alpha_kept = abs(np.corrcoef(X_clean[0], X6[0] - A6[0, 5] * blink)[0, 1])   # against the frontal channel without blinks
print(f"component {blink_comp} has kurtosis {kurt[blink_comp]:.1f} (others {np.round(np.delete(kurt, blink_comp), 1)})")
print(f"blink share of channel variance after cleaning: {100 * residual_blink:.1f}% of before")
print(f"frontal channel after cleaning against the same channel without blinks: r = {alpha_kept:.3f}")

In [ ]:
import numpy as np
def test_blink():
    "the component with the largest kurtosis is the blink"
    assert abs(np.corrcoef(Y6[blink_comp], blink)[0, 1]) > 0.9
def test_clean():
    "after removal, the blink's share of channel power falls by more than 90%, and the alpha rhythm survives"
    assert residual_blink < 0.1 and alpha_kept > 0.95

test_blink()
test_clean()
print("All checks passed")